 # Table of Contents
+ [Import](#Import_0)
+ [Files](#Files_1)
	+ [Input](#Input_2)
	+ [Output](#Output_3)
+ [Parameters](#Parameters_4)
+ [Synonym exploration ](#Synonym_exploration_5)
+ [Parse PubMed journal metadata
](#Parse_PubMed_journal_metadata_6)
+ [Save list](#Save_list_7)


<a class="anchor" id="Import_0"></a>
# <span class=title_0 style="color: #4E2C73">Import</span>

In [1]:
import sys
from pathlib import Path
import ssl
import urllib.request

import pandas as pd

sys.path.append("../..")

from file_management import check_save_file, get_files_dir
from text_analysis import similar_words

<a class="anchor" id="Files_1"></a>
# <span class=title_0 style="color: #4E2C73">Files</span>

<a class="anchor" id="Input_2"></a>
## <span class=title_1 style="color: #622870">Input</span>

This section defines:
- Project input/output directories
- Remote PubMed journal source
- Local keyword lists used to exclude journals

In [2]:
# Resolve project directories
_, INPUT_DIR, OUTPUT_DIR = get_files_dir()

# Remote PubMed journal metadata
JOURNALS_URL = "https://ftp.ncbi.nih.gov/pubmed/J_Medline.txt"

# Local keyword list used to identify journals to avoid
KEYWORDS_AVOID_FILE = Path(INPUT_DIR) / "Journals" / "Avoid" / "key_words_avoid.txt"

# Load avoid keywords
key_words_avoid = pd.read_csv(
    KEYWORDS_AVOID_FILE,
    header=None,
    names=["Words"]
)

# Normalize keywords (strip quotes and lowercase)
key_words_avoid = key_words_avoid["Words"].str.strip("'").str.lower()

<a class="anchor" id="Output_3"></a>
## <span class=title_1 style="color: #622870">Output</span>

The final output is a JSON file containing batched lists of
journal NLM IDs to exclude.

In [3]:
OUTPUT_FILE_NEG = 'list_journals_avoid.json'
OUTPUT_FILE_POS = 'list_journals_possible.json'

<a class="anchor" id="Parameters_4"></a>
# <span class=title_0 style="color: #4E2C73">Parameters</span>

Configuration values controlling batching and network access.

In [4]:
# Number of journal IDs per batch
JOURNAL_BATCH_SIZE = 2000

# SSL context for accessing NCBI FTP over HTTPS
# (certificate verification disabled for compatibility)
SSL_CONTEXT = ssl.create_default_context()
SSL_CONTEXT.check_hostname = False
SSL_CONTEXT.verify_mode = ssl.CERT_NONE

<a class="anchor" id="Synonym_exploration_5"></a>
# <span class=title_0 style="color: #4E2C73">Synonym exploration </span>

Use semantic similarity to identify additional keywords
that may be relevant for excluding journals.
This step does not modify the dataset automatically.

In [5]:
words = [
    "botany", "neuro", "viral", "virus", "virology", "optical",
    "tissue", "biomaterial", "polymers", "nano", "disease",
    "immunity", "cancer", "annals", "injuries", "immunology",
    "clinic", "therapy", "orthopaedic", "healthcare",
    "microscopy", "medical", "neuron", "surgery",
    "oncology", "cardiology", "material"
]

synonyms = 20

for word in words:
    print(
        f"{synonyms} similar words to '{word}':\n\t"
        f"{similar_words(word, synonyms)}\n"
    )

20 similar words to 'botany':
	['botany', 'paleobotany', 'ethnobotany', 'zoology', 'botanic', 'entomology', 'botanist', 'botanica', 'paleobotanist', 'botanique', 'ornithology', 'botanical', 'ichthyology', 'lichenology', 'mycology', 'zool', 'palaeontology', 'ethnobotanist', 'prytany', 'paleontology']

20 similar words to 'neuro':
	['neuro', 'neuropil', 'neurosync', 'neurokinin', 'neuroma', 'neurone', 'neuroses', 'neurontin', 'neuromuscular', 'neurologic', 'neurobehavioral', 'neuron', 'neuroimmune', 'neurovascular', 'neurofibroma', 'neuroleptic', 'neuromas', 'neuromotor', 'neurochaetae', 'neuroanatomy']

20 similar words to 'viral':
	['viral', 'nonviral', 'virale', 'virago', 'arboviral', 'adenoviral', 'retroviral', 'virally', 'lentiviral', 'antiviral', 'proviral', 'antiretroviral', 'virality', 'herpesvirales', 'metapneumovirus', 'baculovirus', 'cytomegalovirus', 'adenovirus', 'norovirus', 'parvovirus']

20 similar words to 'virus':
	['virus', 'poxvirus', 'coroanvirus', 'coronvirus', 'ebo

20 similar words to 'material':
	['material', 'nonmaterial', 'metamaterial', 'immaterial', 'materiales', 'materia', 'biomaterial', 'materiality', 'nanomaterials', 'materializing', 'materially', 'materialising', 'materialistic', 'substructures', 'somateria', 'materialization', 'compo', 'microstructures', 'polysubstance', 'contents']



Additional domain-specific stems manually added to the
keyword list to improve journal exclusion coverage.

In [6]:
EXTRA_KEYWORDS = pd.Series([
    "odonto", "allergo", "nervos", "anaesthes", "anato","histor",
    "management", "chirurgica", "derma", "hepato", "business","metallurg",
    "rheuma", "skin", "urolo", "geronto", "dental","dentis", "wireless",
    "citizen", "nurs", "political economy", "anthropol","psycholog","obstretic",
    "oftalmo","ophtalmol","laryngo", "laringo","magnetic","rehabil", "surgery",
    "childhood", "psychiatr"
])

key_words_avoid = pd.concat(
    [key_words_avoid, EXTRA_KEYWORDS],
    ignore_index=True
).drop_duplicates()

<a class="anchor" id="Parse_PubMed_journal_metadata_6"></a>
# <span class=title_0 style="color: #4E2C73">Parse PubMed journal metadata
</span>


This step scans the `J_Medline.txt` file line by line.
If any avoid keyword appears in a journal block,
the journal's NLM ID is collected.

In [7]:

neg_journals_ids = []
pos_journals_ids = []

avoid_journal = False
show = False
with urllib.request.urlopen(JOURNALS_URL, context=SSL_CONTEXT) as html:
    for raw_line in html:
        line = raw_line.decode().lower()

        # Flag journal for exclusion
        if any(word in line for word in key_words_avoid):
            avoid_journal = True
#            print('avoiding', line)
        # Capture NLM ID
        if "nlmid:" in line:
            nlm_id = line.split(":", 1)[1].strip()

            if avoid_journal:
                neg_journals_ids.append(nlm_id)
            else:
                pos_journals_ids.append(nlm_id)

            # Reset for next journal
            avoid_journal = False

        # Optional: print accepted journal titles
        elif not avoid_journal and "journaltitle" in line and show:
            print(line.strip())


In [8]:
len(neg_journals_ids)

15392

In [9]:
len(pos_journals_ids)

19764

In [10]:
# Batch excluded journal IDs
# Journal IDs are grouped into fixed-size batches to simplify downstream processing
batched_art_ids_neg = [neg_journals_ids[i * JOURNAL_BATCH_SIZE:(i + 1) * JOURNAL_BATCH_SIZE] 
                   for i in range((len(neg_journals_ids) + JOURNAL_BATCH_SIZE- 1) // JOURNAL_BATCH_SIZE )]

# Batch excluded journal IDs
# Journal IDs are grouped into fixed-size batches to simplify downstream processing
batched_art_ids_pos = [pos_journals_ids[i * JOURNAL_BATCH_SIZE:(i + 1) * JOURNAL_BATCH_SIZE] 
                   for i in range((len(pos_journals_ids) + JOURNAL_BATCH_SIZE- 1) // JOURNAL_BATCH_SIZE )]

<a class="anchor" id="Save_list_7"></a>
# <span class=title_0 style="color: #4E2C73">Save list</span>

In [12]:
check_save_file(
    pd.DataFrame(batched_art_ids_neg).T,
    OUTPUT_FILE_NEG,
    "Journals/Avoid",
    input_dir=True
)

Saved file in: /Users/elisamarquez/Documents/PhD/Utimo/ELISER-StrainDesignDB/files/Input/Journals/Avoid/list_journals_avoid.json


In [13]:
check_save_file(
    pd.DataFrame(batched_art_ids_pos).T,
    OUTPUT_FILE_POS,
    "Journals/Keep",
    input_dir=True
)

Saved file in: /Users/elisamarquez/Documents/PhD/Utimo/ELISER-StrainDesignDB/files/Input/Journals/Keep/list_journals_possible.json
